# PranRakshak - Data Preprocessing (Stable Version)

This notebook:
- Loads ICU dataset
- Cleans missing values
- Uses sliding window (time-series)
- Creates strong features
- Outputs train-ready dataset

Note:
No artificial patient grouping is used.

In [9]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [10]:
DATA_PATH = "../data/raw/dataset.csv"

df = pd.read_csv(DATA_PATH)

# 🔥 Use subset for speed (VERY IMPORTANT)
df = df.sample(n=150000, random_state=42).sort_values("Hour").reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (150000, 44)


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,52.00,1,1.0,0.0,-6.25,1,0,119456
1,0,0,114.0,98.0,NaN,NaN,NaN,NaN,13.0,NaN,...,NaN,NaN,63.91,1,NaN,NaN,-142.64,8,0,18573
2,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,77.00,1,NaN,NaN,-10.12,1,0,118414
3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,55.11,1,NaN,NaN,-81.97,1,0,17583
4,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,58.78,0,1.0,0.0,-145.60,1,0,19958


In [11]:
df.info()
df.isnull().sum().sort_values(ascending=False).head(10)

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 44 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        150000 non-null  int64  
 1   Hour              150000 non-null  int64  
 2   HR                135077 non-null  float64
 3   O2Sat             130336 non-null  float64
 4   Temp              50710 non-null   float64
 5   SBP               128138 non-null  float64
 6   MAP               131177 non-null  float64
 7   DBP               103191 non-null  float64
 8   Resp              126888 non-null  float64
 9   EtCO2             5565 non-null    float64
 10  BaseExcess        8156 non-null    float64
 11  HCO3              6258 non-null    float64
 12  FiO2              12519 non-null   float64
 13  pH                10454 non-null   float64
 14  PaCO2             8378 non-null    float64
 15  SaO2              5185 non-null    float64
 16  AST               2456 non-null

Bilirubin_direct    149714
Fibrinogen          148953
TroponinI           148617
Bilirubin_total     147759
Alkalinephos        147566
AST                 147544
Lactate             146027
PTT                 145493
SaO2                144815
EtCO2               144435
dtype: int64

In [12]:
features = [
    "Hour",
    "HR", "O2Sat", "Temp",
    "SBP", "MAP", "Resp",
    "WBC", "Creatinine", "Glucose",
    "Age", "ICULOS",
    "SepsisLabel"
]

df = df[features]

In [13]:
# Forward fill (time-series style)
df = df.ffill()

# Fill remaining NaNs
df = df.fillna(df.median(numeric_only=True))

In [14]:
WINDOW_SIZE = 6   # smaller = faster + better signal

rows = []

cols = ["HR", "O2Sat", "Temp", "SBP", "MAP", "Resp", "WBC", "Creatinine", "Glucose"]

for i in tqdm(range(WINDOW_SIZE, len(df))):
    
    window = df.iloc[i-WINDOW_SIZE:i]
    
    # 🔥 IMPORTANT: Skip if too many missing originally (optional safety)
    if window.isnull().sum().sum() > 0:
        continue
    
    features = {}
    
    for col in cols:
        values = window[col].values
        
        # Core features (simple but effective)
        features[f"{col}_last"] = values[-1]
        features[f"{col}_mean"] = np.mean(values)
        features[f"{col}_std"] = np.std(values)
        features[f"{col}_trend"] = values[-1] - values[0]

    # Static
    features["Age"] = df.iloc[i]["Age"]
    features["ICULOS"] = df.iloc[i]["ICULOS"]

    # Target
    features["SepsisLabel"] = df.iloc[i]["SepsisLabel"]

    rows.append(features)

processed_df = pd.DataFrame(rows)

print("Processed shape:", processed_df.shape)
processed_df.head()

100%|██████████| 149994/149994 [01:43<00:00, 1451.22it/s]


Processed shape: (149994, 39)


,HR_last,HR_mean,HR_std,HR_trend,O2Sat_last,O2Sat_mean,O2Sat_std,O2Sat_trend,Temp_last,Temp_mean,...,Creatinine_mean,Creatinine_std,Creatinine_trend,Glucose_last,Glucose_mean,Glucose_std,Glucose_trend,Age,ICULOS,SepsisLabel
0,114.0,108.833333,11.553018,31.0,98.0,98.0,0.0,0.0,37.0,37.0,...,0.93,0.0,0.0,127.0,127.0,0.0,0.0,48.21,1.0,0.0
1,114.0,114.000000,0.000000,0.0,98.0,98.0,0.0,0.0,37.0,37.0,...,0.93,0.0,0.0,127.0,127.0,0.0,0.0,46.00,1.0,0.0
2,114.0,114.000000,0.000000,0.0,98.0,98.0,0.0,0.0,37.0,37.0,...,0.93,0.0,0.0,127.0,127.0,0.0,0.0,88.00,1.0,0.0
3,114.0,114.000000,0.000000,0.0,98.0,98.0,0.0,0.0,37.0,37.0,...,0.93,0.0,0.0,127.0,127.0,0.0,0.0,71.00,1.0,0.0
4,114.0,114.000000,0.000000,0.0,98.0,98.0,0.0,0.0,37.0,37.0,...,0.93,0.0,0.0,127.0,127.0,0.0,0.0,50.73,1.0,0.0


In [15]:
OUTPUT_PATH = "../data/processed/train.csv"

processed_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: ../data/processed/train.csv


In [16]:
df.columns

Index(['Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp', 'WBC',
       'Creatinine', 'Glucose', 'Age', 'ICULOS', 'SepsisLabel'],
      dtype='str')